# Customer Churn Prediction

This notebook explores the Telco customer churn dataset and checks the data before model building.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

data_path = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(data_path)
df.head()


## Getting familiar with the dataset


In [ ]:
print("Shape:", df.shape)
df.info()


In [ ]:
df.describe(include="all").T


## Checking the data


In [ ]:
missing = df.isna().sum()
print("Missing values:")
print(missing[missing > 0].sort_values(ascending=False))
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())


In [ ]:
print("TotalCharges dtype:", df["TotalCharges"].dtype)
blank_total = df["TotalCharges"].astype(str).str.strip().eq("").sum()
print("Blank TotalCharges values:", blank_total)
df.loc[df["TotalCharges"].astype(str).str.strip().eq(""), ["customerID", "tenure", "TotalCharges"]].head()


## Churn


In [ ]:
churn_counts = df["Churn"].value_counts()
churn_percent = df["Churn"].value_counts(normalize=True).mul(100).round(2)
print(churn_counts)
print(churn_percent)


In [ ]:
sns.countplot(data=df, x="Churn")
plt.title("Churn count")
plt.xlabel("Churn")
plt.ylabel("Customers")
plt.tight_layout()
plt.show()


## Numeric columns


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print(numeric_cols)
df[numeric_cols].hist(figsize=(11, 7), bins=25)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df, x="Churn", y="tenure", ax=axes[0])
axes[0].set_title("Tenure by churn")
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
axes[1].set_title("Monthly charges by churn")
plt.tight_layout()
plt.show()


## Customer groups


In [ ]:
def churn_rate(data, column):
    rates = data.groupby(column)["Churn"].apply(lambda x: (x == "Yes").mean() * 100)
    return rates.sort_values(ascending=False).round(2)

for column in ["Contract", "PaymentMethod", "InternetService", "TechSupport", "OnlineSecurity"]:
    print("\n", column)
    print(churn_rate(df, column))


In [ ]:
pd.crosstab(df["Contract"], df["Churn"], normalize="index").mul(100).round(2)


In [ ]:
pd.crosstab(df["PaymentMethod"], df["Churn"], normalize="index").mul(100).round(2)


## What I found

- The dataset has 7,043 customers and 21 columns.
- `customerID` is an identifier and should not be used as a model feature.
- `TotalCharges` is stored as text and has blank values that need to be handled.
- Churn is not evenly split between the two classes.
- Month-to-month customers have higher churn than customers on longer contracts.
- Customers with shorter tenure tend to churn more often.
- Monthly charges are generally higher among customers who churn.
